Suppose I have a collection of photos scattered across subfolders in a folder. I would like to create a dataset of downsized images together with labels that indicate presence or absence of each of the several objects in each of the photos. Next I'd like to use CNN using TensorFlow to train a deep learning model, whose input is a downsized image and its output is a vector of 0 or 1 for each of the tracked objects. Give me Python code that covers all stages from collecting images to making predictions.

In [5]:
from pathlib import Path
import pandas as pd
import os
import cv2
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
#import matplotlib.pyplot as plt
import json

# CNN validation:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

## 1. Collect all images

In [2]:
os.chdir("C:/Sereda/Job/portfolio/Python/photo_object_detection")

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

root = Path("C:/Sereda/Foto/2025/09")
if os.path.isdir("data"):
    print("Data folder exists")
else:
    os.mkdir("data")

rows = []
objects = ["V","Y","L","D","S","snow","our_house","cave"]

Data folder exists


In [107]:
for file in root.rglob("*"): # Recursively list all existing files and folders matching the pattern
    if file.suffix.lower() in IMAGE_EXTENSIONS:
        dct = {"filename": str(file)}
        for x in objects:
           dct[x]=0
        rows.append(dct)

df = pd.DataFrame(rows)

df.to_csv("data/labels.csv", index=False)

print(df.head())

                                            filename  V  Y  L  D  S  snow  \
0  C:\Sereda\Foto\2025\09\20250901_185428 D prieh...  0  0  0  0  0     0   
1         C:\Sereda\Foto\2025\09\20250901_185431.JPG  0  0  0  0  0     0   
2  C:\Sereda\Foto\2025\09\20250930_144042 upakovk...  0  0  0  0  0     0   
3  C:\Sereda\Foto\2025\09\01_Donnehue_cave\202509...  0  0  0  0  0     0   
4  C:\Sereda\Foto\2025\09\01_Donnehue_cave\202509...  0  0  0  0  0     0   

   our_house  cave  
0          0     0  
1          0     0  
2          0     0  
3          0     0  
4          0     0  


Initially every label is zero.

## 2. Annotation Tool

In [3]:
def show_image(filename):
    # Display image in a separate window
    img = cv2.imread(filename)
    if img is None:
        raise FileNotFoundError(filename)

    # Resize: width=150, height=100
    img = cv2.resize(img, (150, 100))

    cv2.imshow("Image",img)
    cv2.waitKey(0)

In [4]:
df = pd.read_csv("data/labels.csv")

# Create the figure only once
fig, ax = plt.subplots(figsize=(6, 4))

for i,row in df.iterrows():
    print(row.filename)
    show_image(row.filename)
    # Display image in a separate window
    #cv2.imshow("Image",img)
    #cv2.waitKey(0) # see image in a separate window
    
    print("""
1 V
2 Y
3 L
4 D
5 S
6 snow
7 our_house
8 cave

Press Enter when finished.
""")

    labels = input("Objects: ").split()

    for label in labels:
        idx = int(label)-1
        df.loc[i,objects[idx]]=1

    cv2.waitKey(1)
plt.close(fig)
df.to_csv("data/labels2.csv",index=False)

NameError: name 'plt' is not defined

Typing 1 3 5 means "V"=1, "L"=1, and "S"=1, and the rest are 0.

## 3. Load Dataset

In [6]:
IMG_SIZE = (150,100)
df = pd.read_csv("data/labels.csv")
paths = df.filename.values
labels = df.iloc[:,1:].values

In [7]:
# Image loader
def load_image(path,label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img,channels=3)
    img = tf.image.resize(img,IMG_SIZE)
    img = img/255.
    return img,label

In [112]:
# Build dataset
dataset = tf.data.Dataset.from_tensor_slices(
    (paths,labels)
)
# `from_tensor_slices()` creates a dataset whose elements are one (path, label) pair at a time.
dataset = dataset.map(load_image) # `map()` applies function `load_image` to every dataset element.
# TensorFlow performs the loading lazily: only the images needed for the next training batch are loaded.
dataset = dataset.shuffle(1000)
# Random elements are drawn from this buffer while new examples refill it. This provides good randomness without requiring enough memory for the entire dataset.
dataset = dataset.batch(32) # Neural networks are almost always trained on mini-batches, not single images.
dataset = dataset.prefetch(tf.data.AUTOTUNE) # Performance optimization: During image loading, the GPU does not sit idle, loading and training overlap.
# AUTOTUNE lets TensorFlow automatically determine how many batches to prepare in advance.

This tf.data pipeline is highly scalable: whether you have 500 images or 5 million images, TensorFlow reads, preprocesses, batches, and feeds them to the model incrementally instead of loading everything into memory. It is the standard approach for training deep learning models efficiently.

## 4. CNN Model

In [113]:
model = tf.keras.Sequential([

layers.Conv2D(32,3,activation="relu"),
layers.MaxPooling2D(),

layers.Conv2D(64,3,activation="relu"),
layers.MaxPooling2D(),

layers.Conv2D(128,3,activation="relu"),
layers.MaxPooling2D(),

layers.Flatten(),

layers.Dense(512,activation="relu"),

layers.Dense(len(objects),activation="sigmoid") # Each output is an independent probability.

])

## 5. Compile

In [114]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Binary cross-entropy is the correct loss for multi-label classification.

## 6. Train

In [115]:
history = model.fit(
    dataset,
    epochs=30
)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 190ms/step - accuracy: 0.4715 - loss: 0.4348  
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 189ms/step - accuracy: 0.6504 - loss: 0.2743
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 191ms/step - accuracy: 0.8455 - loss: 0.1936
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 186ms/step - accuracy: 0.8943 - loss: 0.1194
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 213ms/step - accuracy: 0.8699 - loss: 0.0930
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 190ms/step - accuracy: 0.9024 - loss: 0.0689
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 195ms/step - accuracy: 0.9268 - loss: 0.0421
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 187ms/step - accuracy: 0.9187 - loss: 0.0281
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 190ms/step - accuracy: 0.9350 - loss: 0.0176
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 187ms/step - accuracy: 0.9512 - loss: 0.0126
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 195ms/step - accuracy: 0.9512 - loss: 0.0051
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 183ms/step - accuracy: 0.9350 - 

## 7. Save and Load

In [116]:
if os.path.isdir("models"):
    print("Model folder exists")
else:
    os.mkdir("models")

model.save("models/object_model.keras") # Save model

Model folder exists


In [8]:
model = tf.keras.models.load_model( #Load model
    "models/object_model.keras"
)

In [9]:
type(model)

keras.src.models.sequential.Sequential

## 8. Predict

In [123]:
img = tf.io.read_file("test.jpg")
img = tf.image.decode_jpeg(img,channels=3)
img = tf.image.resize(img,IMG_SIZE)
img = img/255.
img = np.expand_dims(img,0)
prediction = model.predict(img)[0]
#objects = ["cat","dog","car","tree","person"]
print("====== Predicted probabilities: =====")
for obj,p in zip(objects,prediction):
    print(obj,p)
print("====== Predicted objects: =====")
for i,obj in enumerate(objects):
    if prediction[i] >= 0.5:
        print(obj)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
====== Predicted probabilities: =====
V 1.9215528e-05
Y 5.355155e-13
L 8.6500185e-09
D 0.00013393599
S 0.999941
snow 0.9999977
our_house 2.9078976e-06
cave 0.00021676073
====== Predicted objects: =====
S
snow


## 9. Validate

For a multi-label CNN (7 independent objects: "V","Y","L","D","S","snow","our_house"), the appropriate metrics are not just accuracy. You should evaluate:
* Binary accuracy: fraction of all object predictions correct.
* Precision / Recall / F1-score per object.
* Macro F1: treats all objects equally.
* Micro F1: aggregates all object decisions (useful for imbalanced datasets).
* ROC-AUC per object (if probabilities are available).
* Confusion matrix per object.

In [15]:
def load_imag(filename, img_size=IMG_SIZE):
    """
    Load and preprocess image exactly as during training.
    """
    img = cv2.imread(filename)

    if img is None:
        raise FileNotFoundError(filename)

    img = cv2.resize(img, img_size)

    # OpenCV BGR -> RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Same normalization as training
    img = img.astype("float32") / 255.0

    return img

In [16]:
def validate_model(model, csv_file, threshold=0.5):
    """
    Validate multi-label CNN model.

    Returns:
        y_true: ground truth labels
        y_prob: predicted probabilities
        y_pred: predicted binary labels
    """

    df = pd.read_csv(csv_file)

    X = []
    y_true = []

    # Load validation images
    for _, row in df.iterrows():

        img = load_imag(row.filename)

        X.append(img)

        labels = row[objects].values.astype("float32")
        y_true.append(labels)

    X = np.array(X)
    y_true = np.array(y_true)

    print("Images loaded:", X.shape)

    # CNN prediction
    y_prob = model.predict(X, verbose=1)

    # Convert probabilities to 0/1
    y_pred = (y_prob >= threshold).astype(int)

    return y_true, y_prob, y_pred

In [17]:
# ----------------------------
# Run validation
# ----------------------------

y_true, y_prob, y_pred = validate_model(
    model,
    "data/validation_labels.csv",
    threshold=0.5
)

Images loaded: (49, 100, 150, 3)
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step


### Overall metrics
For all objects collectively.

In [18]:
print("\n=== Overall Metrics ===")

print(
    "Binary accuracy:",
    accuracy_score(
        y_true.flatten(),
        y_pred.flatten()
    )
)

print(
    "Micro precision:",
    precision_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )
)

print(
    "Micro recall:",
    recall_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )
)

print(
    "Micro F1:",
    f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )
)

print(
    "Macro F1:",
    f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
)


=== Overall Metrics ===
Binary accuracy: 0.7959183673469388
Micro precision: 0.1836734693877551
Micro recall: 0.1836734693877551
Micro F1: 0.1836734693877551
Macro F1: 0.05374361883153715


### Metrics for each object separately

In [19]:
print("\n=== Per Object Performance ===")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=objects,
        zero_division=0
    )
)


=== Per Object Performance ===
              precision    recall  f1-score   support

           V       0.00      0.00      0.00         1
           Y       0.00      0.00      0.00         1
           L       1.00      0.14      0.24        36
           D       0.00      0.00      0.00         5
           S       0.00      0.00      0.00         2
        snow       0.00      0.00      0.00         0
   our_house       0.00      0.00      0.00         0
        cave       0.10      1.00      0.19         4

   micro avg       0.18      0.18      0.18        49
   macro avg       0.14      0.14      0.05        49
weighted avg       0.74      0.18      0.19        49
 samples avg       0.14      0.16      0.15        49



### ROC-AUC per object

Since CNN outputs probabilities, this is useful:

In [20]:
print("\n=== ROC-AUC ===")

for i, obj in enumerate(objects):
    auc = roc_auc_score(
        y_true[:, i],
        y_prob[:, i]
    )

    print(f"{obj:12s}: {auc:.3f}")


=== ROC-AUC ===
V           : 0.958
Y           : 0.312
L           : 0.979
D           : 0.959
S           : 1.000
snow        : nan
our_house   : nan
cave        : 0.967


C:\Users\sered\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\sered\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Confusion matrices

In [21]:
print("\n=== Confusion Matrices ===")

for i, obj in enumerate(objects):

    cm = confusion_matrix(
        y_true[:, i],
        y_pred[:, i]
    )

    print("\n", obj)
    print(cm)


=== Confusion Matrices ===

 V
[[48  0]
 [ 1  0]]

 Y
[[48  0]
 [ 1  0]]

 L
[[13  0]
 [31  5]]

 D
[[44  0]
 [ 5  0]]

 S
[[47  0]
 [ 2  0]]

 snow
[[44  5]
 [ 0  0]]

 our_house
[[49]]

 cave
[[10 35]
 [ 0  4]]


C:\Users\sered\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


## Improvements

### Optimize threshold per object

Using `threshold=0.5` for all objects is often suboptimal. After validation, one can find the best threshold separately:

In [22]:
thresholds = {}

for i, obj in enumerate(objects):
    best_f1 = 0
    best_t = 0.5

    for t in np.arange(0.1, 0.91, 0.05):
        pred = (y_prob[:, i] >= t).astype(int)

        f1 = f1_score(
            y_true[:, i],
            pred,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds[obj] = best_t

print(thresholds)

{'V': 0.5, 'Y': 0.5, 'L': np.float64(0.1), 'D': 0.5, 'S': 0.5, 'snow': 0.5, 'our_house': 0.5, 'cave': np.float64(0.9000000000000002)}


In [23]:
thresholds = {'V': 0.5, 'Y': 0.5, 'L': 0.1, 'D': 0.5, 'S': 0.5, 'snow': 0.5, 'our_house': 0.5, 'cave': 0.9}

### Apply optimized thresholds in model validation

In [24]:
def apply_thresholds(y_prob, thresholds):
    """
    Convert probabilities into binary predictions using
    a different threshold for each class.
    """

    y_pred = np.zeros_like(y_prob, dtype=int)

    for i, obj in enumerate(objects):
        y_pred[:, i] = (
            y_prob[:, i] >= thresholds[obj]
        ).astype(int)

    return y_pred

In [26]:
# Apply thresholds
y_pred = apply_thresholds(y_prob, thresholds)

In [29]:
#######################################################
# Final evaluation
#######################################################

print("\nOverall metrics\n")

print(
    "Binary accuracy:",
    accuracy_score(
        y_true.flatten(),
        y_pred.flatten()
    )
)

print(
    "Micro Precision:",
    precision_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )
)

print(
    "Micro Recall:",
    recall_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )
)

print(
    "Micro F1:",
    f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )
)

print(
    "Macro F1:",
    f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
)

print("\nPer-class report\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=objects,
        digits=3,
        zero_division=0
    )
)


Overall metrics

Binary accuracy: 0.8877551020408163
Micro Precision: 0.5555555555555556
Micro Recall: 0.5102040816326531
Micro F1: 0.5319148936170213
Macro F1: 0.13558352402745993

Per-class report

              precision    recall  f1-score   support

           V      0.000     0.000     0.000         1
           Y      0.000     0.000     0.000         1
           L      1.000     0.583     0.737        36
           D      0.000     0.000     0.000         5
           S      0.000     0.000     0.000         2
        snow      0.000     0.000     0.000         0
   our_house      0.000     0.000     0.000         0
        cave      0.211     1.000     0.348         4

   micro avg      0.556     0.510     0.532        49
   macro avg      0.151     0.198     0.136        49
weighted avg      0.752     0.510     0.570        49
 samples avg      0.469     0.490     0.476        49



### Saving and loading the optimized thresholds

In [30]:
with open("models/thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=4)

In [31]:
thresholds

{'V': 0.5,
 'Y': 0.5,
 'L': 0.1,
 'D': 0.5,
 'S': 0.5,
 'snow': 0.5,
 'our_house': 0.5,
 'cave': 0.9}

Later, when predicting on new images:

In [35]:
with open("models/thresholds.json") as f:
    thresholds = json.load(f)

In [48]:
img = tf.io.read_file("test.jpg")
img = tf.image.decode_jpeg(img,channels=3)
img = tf.image.resize(img,IMG_SIZE)
img = img/255.
img = np.expand_dims(img,0)
prediction = model.predict(img)[0]
#objects = ["cat","dog","car","tree","person"]
print("====== Predicted probabilities: =====")
for obj,p in zip(objects,prediction):
    print(obj,p)
print("====== Predicted objects: =====")
for i,obj in enumerate(objects):
    if prediction[i] >= thresholds[objects[i]]:
        print(obj)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
====== Predicted probabilities: =====
V 1.9215528e-05
Y 5.355155e-13
L 8.6500185e-09
D 0.00013393599
S 0.999941
snow 0.9999977
our_house 2.9078976e-06
cave 0.00021676073
====== Predicted objects: =====
S
snow
